In [1]:
import sys
import pyspark.sql.functions as F

from src.config import data_paths
from src.data_ingestion import DataLoader,DataMerger
from src.logger import logger
from src.exception import M5Exception

In [2]:
#DataLoader initialization
try:
    data_loader_obj = DataLoader(data_paths)
    logger.info("DataLoader initialized successfully!")
except Exception as e:
    logger.info("DataLoader failed!")
    raise M5Exception(str(e),sys) from None


In [3]:
#get input raw datasets
try:
    input_data = data_loader_obj.get_input_data()
    logger.info("data_loader_obj.get_input_data ran successfully!")
except Exception as e:
    logger.info("data_loader_obj.get_input_data run failed!")
    raise M5Exception(str(e),sys) from None

In [4]:
input_data

{'sales_wide': DataFrame[id: string, item_id: string, dept_id: string, cat_id: string, store_id: string, state_id: string, d_1: string, d_2: string, d_3: string, d_4: string, d_5: string, d_6: string, d_7: string, d_8: string, d_9: string, d_10: string, d_11: string, d_12: string, d_13: string, d_14: string, d_15: string, d_16: string, d_17: string, d_18: string, d_19: string, d_20: string, d_21: string, d_22: string, d_23: string, d_24: string, d_25: string, d_26: string, d_27: string, d_28: string, d_29: string, d_30: string, d_31: string, d_32: string, d_33: string, d_34: string, d_35: string, d_36: string, d_37: string, d_38: string, d_39: string, d_40: string, d_41: string, d_42: string, d_43: string, d_44: string, d_45: string, d_46: string, d_47: string, d_48: string, d_49: string, d_50: string, d_51: string, d_52: string, d_53: string, d_54: string, d_55: string, d_56: string, d_57: string, d_58: string, d_59: string, d_60: string, d_61: string, d_62: string, d_63: string, d_64

In [5]:
input_data['sales_wide'].count()

30490

In [6]:
#Data Transformer initialization
try:
    data_merger_obj = DataMerger(input_data)
    logger.info("DataMerger obj initialized successfully!")
except Exception as e:
    logger.info("DataMerger failed!")
    raise M5Exception(str(e),sys) from None

In [21]:
#merge raw datasets
try:
    sales_long, calendar,price,joined_data = data_merger_obj.merge_raw_datasets()
    logger.info("data_merger_obj.merge_raw_datasets ran successfully!")
except Exception as e:
    logger.info("data_merger_obj.merge_raw_datasets run failed!")
    raise M5Exception(str(e),sys) from None

In [22]:
#data preprocessor

joined_data = joined_data.withColumn('year_month',F.concat(F.col('year'),F.lpad('month',2,'0')))\
                            .withColumn('quarter',F.quarter('date'))\
                            .withColumn('quarter',F.lpad('quarter',2,'0'))\
                            .withColumn('year_week',F.concat(F.col('year'),F.lpad(F.weekofyear('date'),2,'0')))



In [23]:
def normalize_event_name(event):
    return (
        event.lower()
        .replace(" ", "_")
        .replace("'", "")
        .replace("-", "_")
    )

In [24]:
event1_values = [
    r["event_name_1"]
    for r in joined_data.select("event_name_1").distinct().collect()
    if r["event_name_1"] is not None
]

event2_values = [
    r["event_name_2"]
    for r in joined_data.select("event_name_2").distinct().collect()
    if r["event_name_2"] is not None
]

all_events = set(event1_values) | set(event2_values)

for event in all_events:

    normalized_event = normalize_event_name(event)

    joined_data = joined_data.withColumn(
        f"f_event_{normalized_event}",
        F.when(
            (F.col("event_name_1") == event) |
            (F.col("event_name_2") == event),
            1
        ).otherwise(0)
    )

In [25]:
joined_data.show(truncate=False)

+--------+--------+-------------+----+------------------+---------+-------+--------+-----+----------+---------+----+-----+----+-------------+------------+------------+------------+-------+-------+-------+----------+----------+-------+---------+-----------------+-------------------------+-------------------+-------------------+-----------------+----------------------+-----------------+-----------------+--------------------+-------------------+---------------------------+-----------------------+----------------+-------------------+--------------+--------------------+-----------------+---------------+------------------+----------------------+-------------------+----------------------+-----------------+-------------------+-----------------+---------------------+---------------------+---------------------+---------------------+--------------------+
|wm_yr_wk|store_id|item_id      |d   |id                |dept_id  |cat_id |state_id|sales|date      |weekday  |wday|month|year|event_name_1 |eve